In [1]:
import gurobipy as gp
from gurobipy import GRB
import os
import pandas as pd
directorio=r"C:\Users\melco\OneDrive\Escritorio\pasantia_imple\datos_16"
d={}
B_num={}
p={1:869.434,2:1533.228,3:2430.901,4:3169.766,5:3947.629,6:4690.218,7:5514.84,8:6051.184,9:6693.546,
   10:7264.281,11:8095.383,12:8530.091,13:9088.648,14:9600.03,15:10486.354,16:11616.937,17:12249.156,
   18:12605.908,19:14312.13,20:15063.13,21:16036.058,22:16509.946,23:17643.719,24:18142.476,25:19010.222}
idx_p=gp.tuplelist(p)
maxi=0


for j, archivo in enumerate(os.listdir(directorio)):
    if archivo.endswith('.csv'):
        cont=0
        ruta_archivo = os.path.join(directorio, archivo)
        df = pd.read_csv(ruta_archivo)
        df['Distance'] = df['Distance'].fillna(0)
        df['dist_Acum'] = df['Distance'].cumsum()
        filtered_df = df[df['Speed[m/s]']<=3.03]
        aux=df['dist_Acum'].iloc[-1]
        if aux>=maxi:
            maxi=aux
        k=0
        for i, row in filtered_df.iterrows():
            k=k+1
            d[(j+1, k)] = row['dist_Acum']
            cont=cont+1
            B_num[j+1]=cont
        #print(j)

B = {}

M=maxi
# Iterar sobre el diccionario original
for clave, valor in d.items():
    primer_valor = clave[0]
    if primer_valor not in B:
        B[primer_valor] = {}
    B[primer_valor][clave] = valor
    
B_max=max(list(B_num.values()))
print(B_max)

print(M)
idx_d=gp.tuplelist(d)
#print(d)
#print(p)
#print(B[1])
print(B_num)
idx_B=gp.tuplelist(B_num)
print(idx_B)
#print(idx_d)
S=sum(B_num.values())
print(S)

1200
39223.01375818591
{1: 951, 2: 1098, 3: 870, 4: 830, 5: 824, 6: 861, 7: 842, 8: 1200, 9: 926, 10: 797, 11: 1183, 12: 672, 13: 1149, 14: 1182, 15: 670, 16: 803}
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
14858


In [5]:
########### 1 FASE #############
m = gp.Model()
x=m.addVars(idx_d,idx_p, vtype = GRB.BINARY, name="x")
y=m.addVars(idx_B, vtype=GRB.CONTINUOUS, name="y")
z=m.addVars(idx_d,vtype=GRB.CONTINUOUS, name="z", ub=300)
alpha=m.addVar(vtype=GRB.CONTINUOUS, name="alpha",ub=1)
m.setObjective(alpha, GRB.MAXIMIZE)#-x.sum("*","*","*")
S=sum(B_num.values())
m.addConstrs((x.sum(i,"*","*")>=25*alpha for i in idx_B),"max_base")
m.addConstrs((x.sum(i,j,"*")<=1 for (i,j) in idx_d ),"max_punto") #for i in idx_B for j in range(1,B_max+1)
m.addConstrs((z[i,j]>=d[i,j]+y[i]-p[k]-M*(1-x[i,j,k]) for (i,j) in idx_d for k in idx_p),"cota_inf")
m.addConstrs((z[i,j]>=-d[i,j]-y[i]+p[k]-M*(1-x[i,j,k]) for (i,j) in idx_d for k in idx_p),"cota_sup")
#m.addConstr(x.sum("*","*","*")>=(0.1*S),"num_datos")
m.addConstrs((x.sum(i,"*",k)<=1 for i in idx_B for k in idx_p),"unicidad")
m.Params.TimeLimit = 1000
m.Params.NoRelHeurTime = 600
m.optimize()

Set parameter TimeLimit to value 1000
Set parameter NoRelHeurTime to value 600
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1255U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 758174 rows, 386325 columns and 3343066 nonzeros
Model fingerprint: 0x77bb5d1e
Variable types: 14875 continuous, 371450 integer (371450 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 3e+02]
  RHS range        [1e+00, 8e+04]
Found heuristic solution: objective -0.0000000
Presolve removed 298644 rows and 14858 columns (presolve time = 6s) ...
Presolve removed 704910 rows and 358052 columns
Presolve time: 5.75s
Presolved: 53264 rows, 28273 columns, 176786 nonzeros
Variable types: 17 continuous, 28256 integer (28256 binary)
Starting NoRel heuristic
Found heuristic solution:

In [7]:
############## 2 FASE #################
m = gp.Model()
x=m.addVars(idx_d,idx_p, vtype = GRB.BINARY, name="x")
y=m.addVars(idx_B, vtype=GRB.CONTINUOUS, name="y")
z=m.addVars(idx_d,vtype=GRB.CONTINUOUS, name="z", ub=300)
m.setObjective(z.sum("*","*"), GRB.MINIMIZE)#-x.sum("*","*","*")
S=sum(B_num.values())
m.addConstrs((x.sum(i,"*","*")>=25*0.72 for i in idx_B),"max_base")
m.addConstrs((x.sum(i,j,"*")<=1 for (i,j) in idx_d ),"max_punto") #for i in idx_B for j in range(1,B_max+1)
m.addConstrs((z[i,j]>=d[i,j]+y[i]-p[k]-M*(1-x[i,j,k]) for (i,j) in idx_d for k in idx_p),"cota_inf")
m.addConstrs((z[i,j]>=-d[i,j]-y[i]+p[k]-M*(1-x[i,j,k]) for (i,j) in idx_d for k in idx_p),"cota_sup")
#m.addConstr(x.sum("*","*","*")>=(0.1*S),"num_datos")
m.addConstrs((x.sum(i,"*",k)<=1 for i in idx_B for k in idx_p),"unicidad")
m.Params.TimeLimit = 1000
m.Params.NoRelHeurTime = 600
m.optimize()

Set parameter TimeLimit to value 1000
Set parameter NoRelHeurTime to value 600
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1255U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 758174 rows, 386324 columns and 3343050 nonzeros
Model fingerprint: 0xe1b4c5ff
Variable types: 14874 continuous, 371450 integer (371450 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 3e+02]
  RHS range        [1e+00, 8e+04]
Found heuristic solution: objective 41412.177575
Presolve removed 295720 rows and 0 columns (presolve time = 6s) ...
Presolve removed 699647 rows and 350556 columns
Presolve time: 6.41s
Presolved: 58527 rows, 35768 columns, 238653 nonzeros
Variable types: 7512 continuous, 28256 integer (28256 binary)
Starting NoRel heuristic
Found heuristic solution: